In [ ]:
from asapdiscovery.data.services.fragalysis.fragalysis_reader import FragalysisFactory
from asapdiscovery.modeling.protein_prep import PreppedComplex
import pandas as pd
from pathlib import Path
from itertools import combinations

In [ ]:
old_fragalysis_directory = "/data1/choderaj/paynea/asap-datasets/full_cross_dock_v2/mpro_fragalysis-04-01-24_curated"

In [ ]:
ff = FragalysisFactory(parent_dir=old_fragalysis_directory)

In [ ]:
plcs = ff.load()

In [ ]:
len(plcs)

In [ ]:
records = []
for c in plcs:
    records.append({"SMILES": c.ligand.smiles,
                    "Compound_Name": c.ligand.compound_name,
                    "Target_Name": c.target.target_name})

In [ ]:
df = pd.DataFrame.from_records(records)

In [ ]:
df.nunique()

## duplication problem:
### SMILES           498
### Compound_Name    515
### Target_Name      543

# Load Prepped

In [ ]:
prepped_path = Path("/data1/choderaj/paynea/asap-datasets/full_cross_dock/mpro_fragalysis-04-01-24_curated_cache_fixed")

In [ ]:
pcs = [PreppedComplex.from_json_file(f) for f in prepped_path.glob("./*/*.json")]

In [ ]:
records = []
for c in pcs:
    records.append({"SMILES": c.ligand.smiles,
                    "Compound_Name": c.ligand.compound_name,
                    "Target_Name": c.target.target_name})

In [ ]:
df = pd.DataFrame.from_records(records)

In [ ]:
df.nunique()

# Remove covalent

In [ ]:
data = pd.read_csv("/data1/choderaj/paynea/asap-datasets/full_cross_dock_v2/mpro_fragalysis-04-01-24_curated/extra_files/Mpro_compound_tracker_csv.csv")

In [ ]:
relevant_data = data[data["Compound ID"].isin(df.Compound_Name.unique())] 

In [ ]:
suspected_covalent = relevant_data[relevant_data.why_suspected_SMILES == "Covalent"]["Compound ID"].unique()

In [ ]:
len(suspected_covalent)

remove 85 covalent Compound IDs

In [ ]:
noncovalent = df[~df.Compound_Name.isin(suspected_covalent)]

### how many target names were removed?

In [ ]:
covalent_target_names = set(df.Target_Name.unique()) - set(noncovalent.Target_Name.unique()) 

In [ ]:
len(covalent_target_names)

## add Date info

In [ ]:
soaks_path = Path(old_fragalysis_directory) / "extra_files" / "Mpro_soaks.csv"

In [ ]:
soaks = pd.read_csv(soaks_path)

In [ ]:
from datetime import datetime

In [ ]:
def process_crystal_data(soaks):
    ddf = soaks.loc[:, ["Sample Name", "Data Collection Date"]]
    ddf["Sanitized_Date"] = ddf["Data Collection Date"].apply(date_processor)
    ddf.columns = ["Structure_Name", "Data_Collection_Date", "Structure_Date"]
    date_dict = ddf.set_index("Structure_Name").to_dict()["Structure_Date"]
    date_dict = {k: str(v) for k, v in date_dict.items() if str(v) != "NaT"}
    structure_to_cmpd_dict = {
        row["Sample Name"]: row["Compound ID"]
        for idx, row in soaks.iterrows()
        if row["Sample Name"] in date_dict
    }

    return date_dict, structure_to_cmpd_dict

In [ ]:
def date_processor(date_string):
    if type(date_string) == str and not date_string == "None":
        try:
            return datetime.strptime(date_string, "%Y-%m-%d %H:%M:%S")
        except ValueError:
            return datetime.strptime(date_string, "%d/%m/%Y %H:%M")
    else:
        return None

In [ ]:
date_dict, structure_to_cmpd_dict = process_crystal_data(soaks)

In [ ]:
noncovalent["Date"] = noncovalent.Target_Name.apply(lambda x: date_dict.get(x[:-3], None))

In [ ]:
noncovalent[noncovalent.Date.isna()]

### make sure these SMILES with missing dates end up in final count

In [ ]:
dateless = noncovalent[noncovalent.Date.isna()].SMILES

# Check for Duplicates

In [ ]:
from itertools import combinations
def get_duplicates(df):
    return_dict = {}
    for col1, col2 in combinations(df.columns, 2):
        counts = df.groupby(col1).nunique()
        return_dict[f"{col1}_to_{col2}"] = counts[(counts[col2] > 1)].index.unique()
    return return_dict

In [ ]:
get_duplicates(noncovalent)

In [ ]:
deduped = noncovalent.sort_values("Date").groupby("SMILES").head(1)
deduped = deduped.sort_values("Date").groupby("Compound_Name").head(1)

In [ ]:
deduped[deduped.SMILES.isin(dateless)]

In [ ]:
get_duplicates(deduped)

In [ ]:
len(deduped)

In [ ]:
deduped

In [ ]:
missing_smiles = set(df.SMILES.unique()) - set(deduped.SMILES.unique())

In [ ]:
len(missing_smiles)

In [ ]:
missing_cmpd_name = set(df.Compound_Name.unique()) - set(deduped.Compound_Name.unique())

In [ ]:
len(missing_cmpd_name)

In [ ]:
missing_target_name = set(df.Target_Name.unique()) - set(deduped.Target_Name.unique())

In [ ]:
len(missing_target_name)

In [ ]:
missing_target_name_df = df[df.Target_Name.isin(missing_target_name)]

In [ ]:
missing_target_name_df.nunique()

In [ ]:
missing_noncovalent_target_names = missing_target_name - covalent_target_names

In [ ]:
len(missing_noncovalent_target_names)

In [ ]:
missing_noncovalent_target_names_df = df[df.Target_Name.isin(missing_noncovalent_target_names)]

In [ ]:
missing_noncovalent_target_names_df.nunique()

In [ ]:
missing_noncovalent_target_names_smiles = set(missing_noncovalent_target_names_df.SMILES.unique())

In [ ]:
missing_noncovalent_target_names_smiles - set(deduped.SMILES.unique())

In [ ]:
df[df.SMILES == 'Cc1ccncc1NC(=O)Cc2cc(cc(c2)Cl)O[C@H]3CC(=O)N3']

In [ ]:
deduped[deduped.Compound_Name == "TRY-UNI-2eddb1ff-7"]

Ok I've convinced myself that the deduplication here works.
I've removed the covalent molecules, and then deduplicated on the SMILES using the first date of collection.
The only wierd example is `TRY-UNI-2eddb1ff-7`, which has two different stereochemistries associated with it, so two different smiles for the same molecule.
In theory this could have happened with more molecules, but this is the only one I've found where both enantiomers ended up in a crystal structure.
I'm going to just use the first one that was collected (`Mpro-x10789_0A`)